# 01 — Analisi

Carica il dataframe aggregato e produce le viste che servono al capitolo
risultati. Nessuna misura viene rifatta qui: questo notebook legge `results/`,
non tocca le board.

Prima di aprirlo:

```bash
python -m tools.aggregate --out data.parquet
```


In [ ]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 80)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
df = pd.read_parquet(ROOT / "data.parquet")
len(df), df["status"].value_counts().to_dict()


## Copertura della matrice

Le celle `skipped` e `failed` restano nel dataframe: servono a distinguere una
combinazione non supportata da una che è crashata. Aggregare solo i successi
significa non poter dire quante celle sono state davvero misurate.


In [ ]:
coverage = (df.groupby(["board", "backend", "status"]).size()
            .unstack(fill_value=0))
display(coverage)

if (df["status"] == "skipped").any():
    display(df[df.status == "skipped"].groupby("reason").size()
            .sort_values(ascending=False).rename("celle"))


## Stato degli artefatti

Un artefatto `degraded` ha superato le soglie di `quantization.validation`: è
velocissimo e predice altro. Va guardato **prima** dei grafici di latenza,
altrimenti compare come il risultato migliore.

`actual_e2e = False` con `requested_e2e = True` significa che l'export è
ricaduto sulla testa one-to-many, quindi quella latenza include l'NMS.


In [ ]:
cols = [c for c in ["model", "quantization", "backend", "val_status",
                    "val_max_abs_diff_vs_fp32", "val_map50_delta",
                    "val_requested_e2e", "val_actual_e2e",
                    "val_e2e_fallback_reason"] if c in df.columns]
artifacts = df[cols].drop_duplicates()
display(artifacts)

degraded = artifacts[artifacts.get("val_status") == "degraded"]
print(f"artefatti degradati: {len(degraded)}")


## Latenza

Mediana e p99, mai la sola media: su edge, con throttling e governor, la coda
conta più del valore centrale.


In [ ]:
ok = df[df.status == "ok"].copy()
lat = (ok.groupby(["board", "backend", "quantization", "compute_target"])
         [["lat_median_ms", "lat_p99_ms", "lat_mean_ms"]].median()
         .round(3).sort_values("lat_median_ms"))
display(lat)


In [ ]:
pivot = ok.pivot_table(index="backend", columns="quantization",
                       values="lat_median_ms", aggfunc="median")
ax = pivot.plot.bar(figsize=(7, 4))
ax.set_ylabel("latenza mediana [ms]")
ax.set_title("Latenza mediana per backend e quantizzazione")
plt.show()

# guadagno rispetto a fp32, per backend
if "fp32" in pivot.columns:
    display((pivot["fp32"].div(pivot.drop(columns="fp32").T).T).round(3))


## Latenza e accuratezza

La mAP è calcolata una volta per artefatto esportato: non dipende da profilo di
potenza, numero di core o frequenza, quindi lo stesso valore si ripete su tutte
le celle che usano quell'artefatto.


In [ ]:
pts = ok.dropna(subset=["lat_median_ms", "acc_map50"])
fig, ax = plt.subplots(figsize=(6.5, 4.5))
for key, grp in pts.groupby("backend"):
    ax.scatter(grp.lat_median_ms, grp.acc_map50, label=key, s=30)
ax.set_xlabel("latenza mediana [ms]"); ax.set_ylabel("mAP@50")
ax.legend(); ax.set_title("Latenza / accuratezza")
plt.show()


## Contesto termico

Se la latenza cresce con `order_index`, il cooldown non è bastato e le ultime
celle dello sweep sono sistematicamente più lente delle prime. È il motivo per
cui l'indice viene registrato in ogni risultato.

`rt_throttled` diverso da falso significa cella contaminata: sul Pi il firmware
applica un throttling proprio, indipendente dal governor.


In [ ]:
thermal = ok[ok.order_index >= 0]
if len(thermal) > 3:
    ax = thermal.plot.scatter(x="order_index", y="lat_median_ms", figsize=(7, 3.5))
    ax.set_title("Latenza rispetto all'ordine di esecuzione")
    plt.show()
    print("correlazione ordine/latenza:",
          round(thermal[["order_index", "lat_median_ms"]].corr().iloc[0, 1], 3))

if "rt_throttled" in ok.columns:
    display(ok[ok.rt_throttled == True][
        [c for c in ["cell_id", "board", "freq_target", "compute_target",
                     "rt_temp_start_c", "rt_temp_end_c"] if c in ok.columns]])


## Energia e machine hours

In [ ]:
if "energy_mean_power_w" in ok.columns:
    display(ok.groupby(["board", "freq_target", "compute_target"])
              [["energy_mean_power_w", "energy_energy_j"]].mean().round(3))

hours = (df.groupby("time_device")["time_wall_s"].sum() / 3600).round(2)
display(hours.rename("wall_h"))


## Report

```bash
python -m tools.report            # reports/<timestamp>/ + reports/latest
```
